<a href="https://colab.research.google.com/github/owennoimor/bco7006_practice/blob/main/epss_v1_no_chatgpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **VERSION 1 BEFORE CHATGPT**

#**Assessment 3: Programming Solutions**

## Problem Statement:

The current enrolment process at the university is time-consuming, inefficient, and does not effectively match students with appropriate courses. This results in lower satisfaction and retention rates among students, which can harm the university's reputation and financial stability.

## Solution:
To address this problem, the university should implement an enrollment process Screening System (EPSS) that checks the course structure and students' enrolment background before enrolling them. This system can be achieved by leveraging programming algorithms that analyse student data to recommend courses that align with their interests and academic abilities.

# **Part 1: First stage**


Students will develop an Enrollment Management system for a university. In first stage they will solve the problem and coding without using Chat GPT and in second stage they will get support of ChatGPT ; The students need to provide enough evidences to prove their experience such as codes and screenshots;

#Condition 1: Course Structure

 The course structure is submitted below. The unit’s prerequisite for the enrolment should be considered.

- unit_name

- unit_code

- pre-req/ condition

- semest

**Step 1 - Catalogue + one student**

In [ ]:
#minimum example

catalogue={
    "BCO7006":{"name":"Coding for BA","prereqs":[],"semester":1,"female_only":0},
    "BCO7000":{"name":"Biz analytics","prereqs":[],"semester":1,"female_only":0},
    "BCO6008":{"name":"Predictive Analytics","prereqs":["BCO7000"],"semester":2,"female_only":0},
    "BCO7007":{"name":"Machine Learning","prereqs":["BCO7006","BCO7000"],"semester":2,"female_only":0},
    "WOM1000":{"name":"Women in STEM","prereqs":[],"semester":0,"female_only":1}
}

#student sample data
alice={"st_name":"Alice","st_id":"S001","gender":"Female","unit_passed":["BCO7000"]}

In [ ]:
print("Catalogue keys:", list(catalogue.keys()))
print("\nAlice:", alice)
print("\nPrereq for BCO6008:", catalogue["BCO6008"]["prereqs"])

Catalogue keys: ['BCO7006', 'BCO7000', 'BCO6008', 'BCO7007', 'WOM1000']

Alice: {'st_name': 'Alice', 'st_id': 'S001', 'gender': 'Female', 'unit_passed': ['BCO7000']}

Prereq for BCO6008: ['BCO7000']


In [ ]:
#more advanced:
#using dataclass

from dataclasses import dataclass, field
from typing import List

@dataclass
class Unit:
    code: str
    name: str
    prereqs: List[str]=field(default_factory=list)
    semester: int=1
    female_only: int=0 #if 0= open to all

@dataclass
class Student:
    st_name: str
    st_id: str
    gender: str
    unit_passed: List[str]=field(default_factory=list)

CATALOGUE=[
        Unit("BCO7006","Coding for BA",prereqs=[],semester=1,female_only=0),
        Unit("BCO7000","Biz analytics",prereqs=[],semester=1,female_only=0),
        Unit("BCO6008","Predictive Analytics",prereqs=["BCO7006"],semester=2,female_only=0),
        Unit("BCO7007","Machine Learning",prereqs=["BCO7000", "BCO7006"],semester=2,female_only=0),
        Unit("WOM1000","Women in STEM",prereqs=[],semester=0,female_only=1)
    ]

#Condition 2: Students Enrolment background

The students are allowed to enrol in two units however the program should check the students’ enrolment background.
Here is a table as a sample of an input.

- st_name

- st_id

- gender

- unit_passed

- semester

**Step 2 — One screening function**

In [ ]:
def can_enrol(student, unit_code,catalogue):
  """Return (ok,message) for single unit"""
  if unit_code not in catalogue:
    return (False,f"Unit {unit_code} not found")

  unit=catalogue[unit_code]

  #check prereq
  missing=[p for p in unit["prereqs"] if p not in student["unit_passed"]]
  if missing:
    return (False,f"Cannot enrol in {unit_code}. Missing prereqs {','.join(missing)}")

  #check conditions
  if unit["female_only"] and student["gender"].lower()!="female":
    return (False,f"Cannot enrol in {unit_code}. Only for female")
  return (True,f"Enrolment OK for {unit_code}{unit['name']}")


In [ ]:
#test on cases - ideally edge cases
bob = Student(st_name="Bob", st_id="S002", gender="male", unit_passed=[])

print(can_enrol(alice , "BCO7006", catalogue))
print(can_enrol(alice , "XZXXXX", catalogue))
print(can_enrol(bob , "BCO7006", catalogue))

(True, 'Enrolment OK for BCO7006Coding for BA')
(False, 'Unit XZXXXX not found')
(True, 'Enrolment OK for BCO7006Coding for BA')


#Condition 3: Enrolment process
Your program should check if the students are eligible for enrolment, otherwise to send an appropriate message and advice to the student
Enrolment as an output:

- enrol_code

- Unit_code

- St_id

- semester

some units have special consition(s)

**Step 3 — The loop over requested units**

In [ ]:
def process_request(student,requested_units,semester,catalogue,enrolments,next_code):
  """Process an enrolment reques, transform 'enrolments ' and return the next code"""
  print(f"Processing request for {student['st_name']}")

#cap checking =no more than 2
  if len(requested_units>2):
    print(f"Rejecting request for {student['st_name']}")
    return next_code

    for unit_code in requested_units:
      ok,message=can_enrol(student,unit_code,catalogue)
      if ok:
        record={
          "enrol_code":next_code,
          "unit_code":unit_code,
          "st_id":student["st_id"],
          "semester":semester
        }
        enrolments.append(record)
        next_code+=1
        print(f"OK:{message}->enrol_code:{next_code}")
      if not ok:
        print(f"Fail")
        return next_code

**Step 4 — Wrap in a class**

In [ ]:
class EnrolmentSystemMin:
  """Min version of the screening system"""
  MAX_UNITS = 2 #

  def __init__(self, catalogue, enrolments=None, next_code=1001):
    self.catalogue = catalogue # Corrected 'set.catalogue' to 'self.catalogue'
    self.enrolments = enrolments if enrolments is not None else []
    self.next_code = next_code

  def check_student_eligibility(self, student, requested_units):
    """Checks if a student can enrol in a list of units, considering MAX_UNITS and individual unit eligibility.
       Returns a list of (ok, message) tuples for each requested unit.
    """
    results = []
    if len(requested_units) > self.MAX_UNITS:
      return [(False, f"Cannot enrol in more than {self.MAX_UNITS} units. Requested: {len(requested_units)}")]

    for unit_code in requested_units:
      ok, message = can_enrol(student, unit_code, self.catalogue) # Reusing
      results.append((ok, message))
    return results

  def enrol_student(self, student, requested_units, semester):
    """Attempts to enrol a student in the requested units.
       Returns True if all units are successfully enrolled, False otherwise.
    """
    print(f"Processing enrolment request for {student['st_name']}...")
    eligibility_results = self.check_student_eligibility(student, requested_units)

    if len(eligibility_results) == 1 and not eligibility_results[0][0]: # >2
      print(f"Rejected: {eligibility_results[0][1]}")
      return False

    all_ok = True
    for i, (ok, message) in enumerate(eligibility_results):
      unit_code = requested_units[i]
      if ok:
        record = {
            "enrol_code": self.next_code,
            "unit_code": unit_code,
            "st_id": student["st_id"],
            "semester": semester
        }
        self.enrolments.append(record)
        print(f"Enrolment OK for {unit_code}: {message} (Enrol Code: {self.next_code})")
        self.next_code += 1
      else:
        print(f"Enrolment FAILED for {unit_code}: {message}")
        all_ok = False
    return all_ok

#Test scenarios

In [ ]:
# Initialize
epss = EnrolmentSystemMin(catalogue=catalogue)

print("Initial enrolments:", epss.enrolments)



Initial enrolments: []


Scenario 1: Student has all prereqs, requests 2 valid units

In [ ]:
# Create Student - Owen

owen = {"st_name":"Owen","st_id":"S003","gender":"Male","unit_passed":["BCO7000", "BCO7006"]}


print("- Scenario 1 -")
# Expected: both enrol
epss.enrol_student(owen, ["BCO6008", "BCO7007"], semester=1)


- Scenario 1 -
Processing enrolment request for Owen...
Enrolment OK for BCO6008: Enrolment OK for BCO6008Predictive Analytics (Enrol Code: 1001)
Enrolment OK for BCO7007: Enrolment OK for BCO7007Machine Learning (Enrol Code: 1002)


True

Scenario 2: Student missing one prereq

In [ ]:
print("- Scenario 2 -")
# Expected: one enrols, one rejected
epss.enrol_student(alice, ["BCO6008", "BCO7007"], semester=2)

- Scenario 2 -
Processing enrolment request for Alice...
Enrolment OK for BCO6008: Enrolment OK for BCO6008Predictive Analytics (Enrol Code: 1003)
Enrolment FAILED for BCO7007: Cannot enrol in BCO7007. Missing prereqs BCO7006


False

Scenario 3: Student has no prereqs, requests two advanced units

In [ ]:
#Create Student - Bob
bob = {"st_name":"Bob","st_id":"S002","gender":"Male","unit_passed":[]}


print("- Scenario 3 -")
# Expected: both rejected
epss.enrol_student(bob, ["BCO6008", "BCO7007"], semester=2)

- Scenario 3 -
Processing enrolment request for Bob...
Enrolment FAILED for BCO6008: Cannot enrol in BCO6008. Missing prereqs BCO7000
Enrolment FAILED for BCO7007: Cannot enrol in BCO7007. Missing prereqs BCO7006,BCO7000


False

Scenario 4: Female student requests WOM1000

In [ ]:
print("- Scenario 4 -")
# Expected: enrol
epss.enrol_student(alice, ["WOM1000"], semester=2)

- Scenario 4 -
Processing enrolment request for Alice...
Enrolment OK for WOM1000: Enrolment OK for WOM1000Women in STEM (Enrol Code: 1004)


True

Scenario 5: 	Male student requests WOM1000

In [ ]:
print("- Scenario 5 -")
# Expected: rejected
epss.enrol_student(owen, ["WOM1000"], semester=2)

- Scenario 5 -
Processing enrolment request for Owen...
Rejected: Cannot enrol in WOM1000. Only for female


False

Scenario 6: Any student requests 3 units


In [ ]:
print("- Scenario 6 -")
# Expected: Rejected
epss.enrol_student(owen, ["BCO6008", "BCO7007", "WOM1000"], semester=2)


- Scenario 6 -
Processing enrolment request for Owen...
Rejected: Cannot enrol in more than 2 units. Requested: 3


False